In [1]:
# Importazione pacchetti
import pandas as pd
import datetime as dt

In [2]:
# Importazione dataset
pilot = pd.read_csv(r'Project\03. Variabilità di Peso nel Confezionamento Caffè\pilot_test.csv')

# 1. DESCRIZIONE DATASET

In [3]:
pilot.head()

,date,shift,machine,batch_id,weight_grams
0,2026-06-08,Notte,Macchina_2_Riferimento,P2015,249.80
1,2026-06-01,Notte,Macchina_1_Contromisura,P2000,252.29
2,2026-06-06,Notte,Macchina_1_Contromisura,P2010,249.93
3,2026-06-01,Notte,Macchina_2_Riferimento,P2001,246.46
4,2026-06-11,Notte,Macchina_2_Riferimento,P2021,250.02


In [4]:
pilot.info()

<class 'pandas.DataFrame'>
RangeIndex: 282 entries, 0 to 281
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   date          282 non-null    str    
 1   shift         282 non-null    str    
 2   machine       282 non-null    str    
 3   batch_id      282 non-null    str    
 4   weight_grams  279 non-null    float64
dtypes: float64(1), str(4)
memory usage: 22.8 KB


In [5]:
pilot.describe()

,weight_grams
count,279.000000
mean,249.177814
std,2.261303
min,242.470000
25%,248.150000
50%,249.340000
75%,250.550000
max,255.400000


In [6]:
pilot.shape

(282, 5)

# 2. INCONSISTENZE CATEGORIA

In [7]:
# Valori unici colonna shift
pilot['shift'].unique()

<ArrowStringArray>
['Notte']
Length: 1, dtype: str

In [8]:
# Ripristino valori corretti
pilot['shift'] = pilot['shift'].str.strip()
pilot['shift'] = pilot['shift'].str.title()

In [9]:
# Controllo valori corretti in shift
pilot['shift'].unique()

<ArrowStringArray>
['Notte']
Length: 1, dtype: str

# 3. MISSING VALUES

In [10]:
# Valori nulli per colonna
pilot.isnull().sum()

date            0
shift           0
machine         0
batch_id        0
weight_grams    3
dtype: int64

In [11]:
# % valori nulli per colonna
(pilot.isnull().mean()*100).round(1)

date            0.0
shift           0.0
machine         0.0
batch_id        0.0
weight_grams    1.1
dtype: float64

In [12]:
# Righe con valori nulli
pilot_null = pilot[pilot.isnull().any(axis =1)]
pilot_null

,date,shift,machine,batch_id,weight_grams
70,2026-06-05,Notte,Macchina_2_Riferimento,P2009,NaN
131,2026-06-11,Notte,Macchina_2_Riferimento,P2021,NaN
175,2026-06-01,Notte,Macchina_1_Contromisura,P2000,NaN


In [13]:
null_op = pilot_null.groupby(['shift', 'machine'])['weight_grams'].agg(
    mancanti = 'size'
)
null_op['%_null'] = (null_op['mancanti']/null_op['mancanti'].sum()*100).round(2)

null_op

mancanti  %_null
shift machine                                  
Notte Macchina_1_Contromisura         1   33.33
      Macchina_2_Riferimento          2   66.67

In [14]:
# Mediana delle pesate per turno
mediana = pilot.groupby('machine')['weight_grams'].transform('median')
mediana
# Sostituzione dei valori nulli con mediana dei dati
pilot['weight_grams'] = pilot['weight_grams'].fillna(mediana)

In [15]:
pilot['weight_grams'].isnull().sum()

np.int64(0)

# 4. OUTLIERS

Essendo che le pesate sono comprese tra 245 e 255, sono rpesenti outliers. Questi li posso identificare ma la sostituzione dei valori deve essere fatta su numeri estremi, non su valori fuori dal range ma realistici.

In [16]:
# Verifico la presenza di outliers all'interno del dataset
# Quartile 1 
Q1 = pilot['weight_grams'].quantile(0.25)

# Quartile 3
Q3 = pilot['weight_grams'].quantile(0.75)

# Range Interquartile
IQR = Q3-Q1

# Limiti
limite_inferiore = Q1-(IQR*1.5)
limite_superiore = Q3+(IQR*1.5)

outliers = pilot[(pilot['weight_grams']<limite_inferiore) | (pilot['weight_grams']>limite_superiore)]
outliers

,date,shift,machine,batch_id,weight_grams
9,2026-06-08,Notte,Macchina_2_Riferimento,P2015,243.94
23,2026-06-12,Notte,Macchina_2_Riferimento,P2023,243.46
32,2026-06-03,Notte,Macchina_2_Riferimento,P2005,242.47
44,2026-06-11,Notte,Macchina_2_Riferimento,P2021,243.52
100,2026-06-04,Notte,Macchina_2_Riferimento,P2007,242.58
148,2026-06-14,Notte,Macchina_2_Riferimento,P2027,243.68
159,2026-06-12,Notte,Macchina_2_Riferimento,P2023,243.65
186,2026-06-01,Notte,Macchina_2_Riferimento,P2001,244.20
224,2026-06-05,Notte,Macchina_2_Riferimento,P2009,242.67
228,2026-06-06,Notte,Macchina_2_Riferimento,P2011,254.48


In [17]:
# Assegno valore minimo
minimo = 200

# Assegno valore massimo
massimo = 300

# Estraggo valori che si trovano sotto il minimo e sopra il massimo
pilot[(pilot['weight_grams']< minimo) | (pilot['weight_grams'] > massimo)].value_counts()

Series([], Name: count, dtype: int64)

# 5. DUPLICATI

In [18]:
# Conteggio dei valori duplicati
pilot.duplicated().sum()

np.int64(3)

In [19]:
# Eliminazione duplicati
pilot_singole = pilot.drop_duplicates()

In [20]:
pilot_singole.duplicated().sum

<bound method Series.sum of 0      False
1      False
2      False
3      False
4      False
       ...  
277    False
278    False
279    False
280    False
281    False
Length: 279, dtype: bool>

# 6. CORREZIONE FORMATI DATASET

In [21]:
def data_quality_report(df):
    report = pd.DataFrame({
        'tipo': df.dtypes,
        'missing': df.isnull().sum(),
        'missing_%': (df.isnull().mean() * 100).round(2),
        'valori_unici': df.nunique()
    })
    print(report)
    print(f"\nRighe totali: {len(df)}")
    print(f"Duplicati: {df.duplicated().sum()}")

data_quality_report(pilot_singole)


                 tipo  missing  missing_%  valori_unici
date              str        0        0.0            14
shift             str        0        0.0             1
machine           str        0        0.0             2
batch_id          str        0        0.0            28
weight_grams  float64        0        0.0           235

Righe totali: 279
Duplicati: 0


In [22]:
date_originale = pilot_singole['date'].copy()   # backup prima di toccare la colonna

# 1. Primo tentativo
pilot_singole['date'] = pd.to_datetime(date_originale, format='%Y-%m-%d', errors='coerce')

# 2. Righe non riconosciute con il primo formato
mask_fail = pilot_singole['date'].isna()

# 3. Riprova SOLO su quelle righe, con il formato alternativo, leggendo dall'originale
pilot_singole.loc[mask_fail, 'date'] = pd.to_datetime(
    date_originale[mask_fail], format='%d/%m/%Y', errors='coerce'
)

# 4. Verifica: quanti NaT restano DAVVERO (dati non recuperabili con nessuno dei due formati)?
pilot_singole['date'].isna().sum()

np.int64(0)

In [23]:
def data_quality_report(df):
    report = pd.DataFrame({
        'tipo': df.dtypes,
        'missing': df.isnull().sum(),
        'missing_%': (df.isnull().mean() * 100).round(2),
        'valori_unici': df.nunique()
    })
    print(report)
    print(f"\nRighe totali: {len(df)}")
    print(f"Duplicati: {df.duplicated().sum()}")

data_quality_report(pilot_singole)

                        tipo  missing  missing_%  valori_unici
date          datetime64[us]        0        0.0            14
shift                    str        0        0.0             1
machine                  str        0        0.0             2
batch_id                 str        0        0.0            28
weight_grams         float64        0        0.0           235

Righe totali: 279
Duplicati: 0


Pulito il datase e sistemato i formati delle colonne esporto in formato CSV per l'analisi dati

In [24]:
pd.DataFrame.to_csv(pilot_singole, r'C:\Users\nicol\DataAnalysis\Project\03. Variabilità di Peso nel Confezionamento Caffè\Dataset\Clean\pilot_clean.csv', index=False)

In [27]:
pilot_singole.groupby('machine')['weight_grams'].std()

machine
Macchina_1_Contromisura    1.357939
Macchina_2_Riferimento     2.782150
Name: weight_grams, dtype: float64